# CIFAR-10 Network with Ultra-Low Precision PO2 Quantization

This notebook demonstrates how to design a Quantization-Aware Training (QAT) Convolutional Neural Network using **QKeras** targeting the **CIFAR-10** image dataset.

### Core Architectural Concepts Shown in This Example

* **CIFAR-10 Spatial Scaling:** Unlike the grayscale $28 \times 28 \times 1$ MNIST dataset, CIFAR-10 contains complex $32 \times 32 \times 3$ RGB color images. The network uses aggressive 128 and 256 filter channels to combat structural complexity under low precision constraints.
* **Ternary Non-Linearity Scaling (`ternary()`):** Layer `act0_m` uses a ternary activation function. This limits the forward-passing feature maps down to just three explicit values: $\{-1, 0, 1\}$. This is incredibly useful for custom hardware designs (like FPGAs), converting continuous floating-point maps into standard logical control bits.
* **Quantized $\mu$-law Nonlinearity (`quantized_ulaw`):** The output classifier uses logarithmic $\mu$-law quantization. This non-uniform quantization method allocates higher bit-resolution around values closer to zero, allowing fine-grained decision clustering during final logic processing.

### 1. Environment Setup & Dependency Installation
Uncomment the cell below if you need to install the required dependencies in your notebook runtime environment.

In [ ]:
# !pip install qkeras-v3 keras tensorflow

### 2. Imports and Initial Configuration Setup

In [3]:
import os
import numpy as np
import keras.ops.numpy as knp
from keras import ops
from keras.datasets import cifar10
from keras.layers import *
from keras.models import Model
from keras.optimizers import *
from keras.utils import to_categorical

from qkeras import *

# Global training hyperparameters
NB_EPOCH = 50
BATCH_SIZE = 64
VERBOSE = 1
NB_CLASSES = 10
OPTIMIZER = Adam(learning_rate=0.0001)
VALIDATION_SPLIT = 0.1

### 3. Load and Normalize CIFAR-10 Dataset

In [4]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

x_train = x_train.astype(float)
x_test = x_test.astype(float)

x_train /= 255.0
x_test /= 255.0

print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")
print("Sample training labels shape:", y_train.shape)

y_train = to_categorical(y_train, NB_CLASSES)
y_test = to_categorical(y_test, NB_CLASSES)

/Users/mariuskoppel/cms/qkeras/venv/lib/python3.11/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


50000 train samples
10000 test samples
Sample training labels shape: (50000, 1)


### 4. Build Heterogeneous Mixed-Precision Network Topology

In [5]:
x = x_in = Input(x_train.shape[1:], name="input")

x = QActivation("quantized_relu_po2(4,4)", name="acti")(x)

x = QConv2D(
    128,
    (3, 3),
    strides=1,
    kernel_quantizer=quantized_po2(4, 1),
    bias_quantizer=quantized_po2(4, 4),
    bias_range=4,
    name="conv2d_0_m",
)(x)
x = QActivation("ternary()", name="act0_m")(x)
x = MaxPooling2D(2, 2, name="mp_0")(x)

x = QConv2D(
    256,
    (3, 3),
    strides=1,
    kernel_quantizer=quantized_po2(4, 1),
    bias_quantizer=quantized_po2(4, 4),
    bias_range=4,
    name="conv2d_1_m",
)(x)
x = QActivation("quantized_relu(6,2)", name="act1_m")(x)
x = MaxPooling2D(2, 2, name="mp_1")(x)

x = QConv2D(
    128,
    (3, 3),
    strides=1,
    kernel_quantizer=quantized_bits(4, 0, 1),
    bias_quantizer=quantized_bits(4, 0, 1),
    name="conv2d_2_m",
)(x)
x = QActivation("quantized_relu(4,2)", name="act2_m")(x)
x = MaxPooling2D(2, 2, name="mp_2")(x)

x = Flatten()(x)
x = QDense(
    NB_CLASSES,
    kernel_quantizer=quantized_ulaw(4, 0, 1),
    bias_quantizer=quantized_bits(4, 0, 1),
    name="dense",
)(x)
x = Activation("softmax", name="softmax")(x)

model = Model(inputs=[x_in], outputs=[x])
model.summary()

/Users/mariuskoppel/cms/qkeras/qkeras/qconvolutional.py:274: UserWarning: bias_range is deprecated in QConv2D layer.
  warnings.warn("bias_range is deprecated in QConv2D layer.")


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ acti (QActivation)              │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_0_m (QConv2D)            │ (None, 30, 30, 128)    │         3,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ act0_m (QActivation)            │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mp_0 (MaxPooling2D)             │ (None, 15, 15, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1_m (QConv2D)            │ (None, 13, 13, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ act1_m (QActivation)            │ (None, 13, 13, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mp_1 (MaxPooling2D)             │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2_m (QConv2D)            │ (None, 4, 4, 128)      │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ act2_m (QActivation)            │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mp_2 (MaxPooling2D)             │ (None, 2, 2, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (QDense)                  │ (None, 10)             │         5,130 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Activation)            │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 598,922 (2.28 MB)

 Trainable params: 598,922 (2.28 MB)

 Non-trainable params: 0 (0.00 B)

### 5. Compile and Fit Model Optimization

In [6]:
model.compile(
    loss="categorical_crossentropy", optimizer=OPTIMIZER, metrics=["accuracy"]
)

history = model.fit(
    x_train,
    y_train,
    batch_size=BATCH_SIZE,
    epochs=NB_EPOCH,
    initial_epoch=1,
    verbose=VERBOSE,
    validation_split=VALIDATION_SPLIT,
)

Epoch 2/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 55s 77ms/step - accuracy: 0.3298 - loss: 1.8858 - val_accuracy: 0.3896 - val_loss: 1.6941
Epoch 3/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 58s 83ms/step - accuracy: 0.4343 - loss: 1.5645 - val_accuracy: 0.4524 - val_loss: 1.5071
Epoch 4/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 69s 97ms/step - accuracy: 0.4857 - loss: 1.4318 - val_accuracy: 0.4820 - val_loss: 1.4327
Epoch 5/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 76s 108ms/step - accuracy: 0.5146 - loss: 1.3559 - val_accuracy: 0.5060 - val_loss: 1.3932
Epoch 6/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 68s 97ms/step - accuracy: 0.5436 - loss: 1.2827 - val_accuracy: 0.5206 - val_loss: 1.3328
Epoch 7/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 65s 93ms/step - accuracy: 0.5635 - loss: 1.2303 - val_accuracy: 0.5312 - val_loss: 1.3284
Epoch 8/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 55s 78ms/step - accuracy: 0.5852 - loss: 1.1816 - val_accuracy: 0.5540 - val_loss: 1.2644
Epoch 9/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 57s 81ms/step - accuracy: 0.6001 - loss: 1.1389 -

### 6. Sub-Network Target Weight Bound and Activation Profiling

In [7]:
outputs = []
output_names = []

for layer in model.layers:
    if layer.__class__.__name__ in [
        "QActivation",
        "Activation",
        "QDense",
        "QConv2D",
        "QDepthwiseConv2D",
    ]:
        output_names.append(layer.name)
        outputs.append(layer.output)

model_debug = Model(inputs=[x_in], outputs=outputs)
outputs = model_debug.predict(x_train)

print("{:30} {: 8.4f} {: 8.4f}".format("input", knp.min(x_train), knp.max(x_train)))

for n, p in zip(output_names, outputs):
    print(f"{n:30} {knp.min(p): 8.4f} {knp.max(p): 8.4f}", end="")
    layer = model.get_layer(n)
    for idx, weights in enumerate(layer.get_weights()):
        if layer.get_quantizers()[idx] is not None:
            quantized_tensor = layer.get_quantizers()[idx](weights)
            weights = ops.convert_to_numpy(quantized_tensor)
        print(
            f" ({knp.min(weights): 8.4f} {knp.max(weights): 8.4f})", end=""
        )
    print("")

score = model.evaluate(x_test, y_test, verbose=VERBOSE)
print("\nTest loss:", score[0])
print("Test accuracy:", score[1])

1182/1563 ━━━━━━━━━━━━━━━━━━━━ 35s 93ms/step

: 

### 7. Total Resource Metric Breakdown

In [ ]:
print_qstats(model)